# Download de XML de NF-e pelo Meu Danfe

Lê as chaves de acesso de um relatório em Excel. Consulta cada uma no [meudanfe.com.br](https://meudanfe.com.br/) e salva os XMLs numa pasta.

**Como funciona:** abre uma janela real do Chrome/Chromium e repete o que você faria à mão: cola a chave, clica em *Buscar* e depois em *Baixar XML*.

- O site tem uma verificação anti-robô (Cloudflare). Normalmente ela passa sozinha. Se aparecer um desafio na janela, **resolva você mesmo** e o script continua.
- Chaves já baixadas são puladas, então é só rodar de novo para completar o que falhou.
- No final, a célula 5 junta todos os XMLs numa planilha (cabeçalho + itens).

## 1. Bibliotecas

In [8]:
import os
import time
import base64
import getpass
import requests
import pandas as pd

## 2. Configuração

In [10]:
# --- CONFIGURAÇÕES ---
ARQUIVO_EXCEL = "notas-xml.xlsx"
COLUNA_DA_CHAVE = "CHAVE_ACESSO_NOTA"
PASTA_DOWNLOAD = "xmls_baixados"
API = "https://api.meudanfe.com.br/v2"
CUSTO_POR_NOTA = 0.03   # R$ por NF-e ainda não adicionada à sua Área do Cliente

# A Api-Key fica fora do notebook: variável de ambiente MEUDANFE_API_KEY ou digitada agora.
api_key = os.environ.get("MEUDANFE_API_KEY") or getpass.getpass("Cole sua Api-Key do Meu Danfe (não será exibida): ")
HEAD = {"Api-Key": api_key.strip()}

download_dir = os.path.abspath(PASTA_DOWNLOAD)
os.makedirs(download_dir, exist_ok=True)

print(f"Lendo a planilha: {ARQUIVO_EXCEL}...")
df = pd.read_excel(ARQUIVO_EXCEL)
if COLUNA_DA_CHAVE not in df.columns:
    raise ValueError(f"A coluna '{COLUNA_DA_CHAVE}' não foi encontrada. Colunas disponíveis: {list(df.columns)}")

chaves = [str(c).strip() for c in df[COLUNA_DA_CHAVE].dropna()]
chaves = list(dict.fromkeys(chaves))  # remove repetidas mantendo a ordem
invalidas = [c for c in chaves if len(c) != 44]
validas = [c for c in chaves if len(c) == 44]
pendentes = [c for c in validas if not os.path.exists(os.path.join(download_dir, f"{c}.xml"))]

print(f"{len(validas)} chaves válidas | {len(invalidas)} inválidas | {len(validas)-len(pendentes)} já baixadas | {len(pendentes)} a baixar")
print(f"Custo máximo estimado: R$ {len(pendentes)*CUSTO_POR_NOTA:.2f} (cobrado só para notas ainda não adicionadas à sua conta)")
if pendentes and input("Continuar? (s/n): ").strip().lower() != "s":
    raise SystemExit("Cancelado.")

def extrair_xml(resp):
    """Aceita resposta em XML puro, JSON com campo de texto/base64, etc."""
    texto = resp.text.lstrip()
    if texto.startswith("<"):
        return resp.content
    try:
        dados = resp.json()
    except ValueError:
        return None
    candidatos = []
    def varrer(o):
        if isinstance(o, dict):
            for v in o.values(): varrer(v)
        elif isinstance(o, list):
            for v in o: varrer(v)
        elif isinstance(o, str):
            candidatos.append(o)
    varrer(dados)
    for s in candidatos:
        if s.lstrip().startswith("<"):
            return s.encode("utf-8")
        try:
            b = base64.b64decode(s, validate=True)
            if b.lstrip().startswith(b"<"):
                return b
        except Exception:
            pass
    return None

def baixar(chave):
    # 1) garante que a nota está na Área do Cliente (consulta; pode cobrar R$ 0,03)
    for _ in range(30):
        r = requests.put(f"{API}/fd/add/{chave}", headers=HEAD, timeout=60)
        if r.status_code in (401, 402, 403):
            raise PermissionError(f"HTTP {r.status_code}: {r.text[:200]}")
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
        status = r.json().get("status")
        if status == "OK":
            break
        if status in ("NOT_FOUND", "ERROR"):
            raise RuntimeError(f"{status}: {r.json().get('statusMessage')}")
        time.sleep(2)  # WAITING / SEARCHING
    else:
        raise TimeoutError("consulta não terminou a tempo")
    time.sleep(1.1)  # a API bloqueia chamadas da mesma chave em menos de 1 s
    # 2) baixa o XML (grátis)
    r = requests.get(f"{API}/fd/get/xml/{chave}", headers=HEAD, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"download HTTP {r.status_code}: {r.text[:200]}")
    xml = extrair_xml(r)
    if xml is None:
        raise RuntimeError(f"resposta sem XML reconhecível: {r.text[:200]}")
    with open(os.path.join(download_dir, f"{chave}.xml"), "wb") as f:
        f.write(xml)

ok, falhas = 0, {}
for i, chave in enumerate(pendentes, 1):
    try:
        baixar(chave)
        ok += 1
        print(f"[{i}/{len(pendentes)}] OK {chave}")
    except PermissionError as e:
        print(f"Parando: {e}")   # Api-Key inválida, substituída ou saldo insuficiente
        break
    except Exception as e:
        falhas[chave] = f"{type(e).__name__}: {e}"
        print(f"[{i}/{len(pendentes)}] ERRO {chave}: {falhas[chave]}")
    time.sleep(1.1)

print(f"\nFinalizado: {ok} baixadas, {len(falhas)} falhas. Pasta: {download_dir}")
for c, m in falhas.items():
    print(" -", c, m)


Cole sua Api-Key do Meu Danfe (não será exibida): ········
Lendo a planilha: notas-xml.xlsx...
30 chaves válidas | 0 inválidas | 29 já baixadas | 1 a baixar
Custo máximo estimado: R$ 0.03 (cobrado só para notas ainda não adicionadas à sua conta)
Continuar? (s/n): s
[1/1] OK 51250200208286000103550010006446781194062156

Finalizado: 1 baixadas, 0 falhas. Pasta: C:\Users\89821491120\jupyter_notebook\python-sagr-dados\nfe-xml\xmls_baixados
